In [1]:
import pandas as pd

In [20]:
df=pd.read_csv(r'D:\DISPATCHLOAD_2025\PUBLIC_ARCHIVE#DISPATCHLOAD#FILE01#202501010000.csv', skiprows=1)

In [24]:
df=df.iloc[:-1]
df=df[['SETTLEMENTDATE', 'RUNNO',
       'DUID', 'TRADETYPE', 'DISPATCHINTERVAL', 'INTERVENTION',
       'CONNECTIONPOINTID', 'DISPATCHMODE', 'AGCSTATUS', 'INITIALMW',
       'TOTALCLEARED']].copy()

In [26]:
Battery=['QBYNB1','BHB1','DPNTB1','RESS1','RIVNB2','WALGRV1']  # NSW BATTERY


In [29]:
df=df[df['DUID'].isin(Battery)]

In [98]:
import pandas as pd
import os
from os import path
import glob
from pathlib import Path  
import numpy as np
import datetime

In [99]:
input_path=r'D:\DISPATCHLOAD_2025'

In [100]:
l=glob.glob(path.join(input_path, '*.csv'))   

In [101]:
df1=pd.DataFrame()
#Battery=['QBYNB1','BHB1','DPNTB1','RESS1','RIVNB2','WALGRV1']
Battery=['BULBES1','HBESS1','PIBESS1','RANGEB1','VBB1']

for n in range(len(l)):
    df=pd.read_csv(l[n], skiprows=1)
    df=df.iloc[:-1]
    df=df[['SETTLEMENTDATE', 'RUNNO',
           'DUID',  'INTERVENTION','INITIALMW',
           'TOTALCLEARED']].copy()
    df=df[df['DUID'].isin(Battery)]
    df1=pd.concat([df,df1], ignore_index=True)

In [102]:
df1.to_csv(r'D:\Analysis\Battery Pattern\VIC_2025_10_Battery.csv')

In [103]:
df1=pd.read_csv(r'D:\Analysis\Battery Pattern\VIC_2025_10_Battery.csv')

In [104]:
def ToD(df,m):
    
    try:
        df['{}'.format(m)]=pd.to_datetime(df['{}'.format(m)],format='%Y/%m/%d %H:%M:%S')
    except: 
        df['{}'.format(m)]=pd.to_datetime(df['{}'.format(m)],format='%d/%m/%Y %H:%M:%S') 

    df['{}'.format(m)]=df['{}'.format(m)] -datetime.timedelta(minutes=5)
    df['Month']=df['{}'.format(m)].dt.month # monthly information 
    df['CY']=df['{}'.format(m)].dt.year    # calander year information 
    df['Quarter']=df['{}'.format(m)].dt.quarter# quartely information
    df['FY']=df['{}'.format(m)].map(lambda d: d.year + 1 if d.month > 6 else d.year) # Financial year infromation
    df['Hour']=df['{}'.format(m)].dt.hour
    df['Day']=df['{}'.format(m)].dt.day
    df['WD']=df['{}'.format(m)].dt.dayofweek

In [105]:
ToD(df1, 'SETTLEMENTDATE')

In [106]:
df1['Dispatch_MW']=np.where(df1['INITIALMW']>0,df1['INITIALMW'],0)
df1['Load_MW']=np.where(df1['INITIALMW']<0,df1['INITIALMW'],0)

In [107]:
df1

,Unnamed: 0,SETTLEMENTDATE,RUNNO,DUID,INTERVENTION,INITIALMW,TOTALCLEARED,Month,CY,Quarter,FY,Hour,Day,WD,Dispatch_MW,Load_MW
0,0,2025-09-01 00:00:00,1.0,BULBES1,0.0,0.030,0.0,9,2025,3,2026,0,1,0,0.03,0.000
1,1,2025-09-01 00:00:00,1.0,HBESS1,0.0,-0.200,0.0,9,2025,3,2026,0,1,0,0.00,-0.200
2,2,2025-09-01 00:00:00,1.0,PIBESS1,0.0,-2.999,-2.0,9,2025,3,2026,0,1,0,0.00,-2.999
3,3,2025-09-01 00:00:00,1.0,RANGEB1,0.0,-4.150,-3.0,9,2025,3,2026,0,1,0,0.00,-4.150
4,4,2025-09-01 00:00:00,1.0,VBB1,0.0,-13.100,0.0,9,2025,3,2026,0,1,0,0.00,-13.100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391915,391915,2025-01-31 23:55:00,1.0,BULBES1,0.0,-2.480,0.0,1,2025,1,2025,23,31,4,0.00,-2.480
391916,391916,2025-01-31 23:55:00,1.0,HBESS1,0.0,-0.600,0.0,1,2025,1,2025,23,31,4,0.00,-0.600
391917,391917,2025-01-31 23:55:00,1.0,PIBESS1,0.0,-0.374,0.0,1,2025,1,2025,23,31,4,0.00,-0.374
391918,391918,2025-01-31 23:55:00,1.0,RANGEB1,0.0,0.110,0.0,1,2025,1,2025,23,31,4,0.11,0.000


In [108]:
df1.groupby(['DUID','Month','Hour']).agg({'Dispatch_MW':'mean','Load_MW':'mean'}).reset_index().to_csv(r'D:\Analysis\Battery Pattern\Raw Data\VIC_t1.csv')

In [109]:
df1.groupby(['DUID','Month','Day']).agg({'Dispatch_MW':'mean','Load_MW':'mean'}).reset_index().to_csv(r'D:\Analysis\Battery Pattern\Raw Data\VIC_t2.csv')

In [110]:
df1_2=df1.groupby(['DUID','Month','Day']).agg({'Dispatch_MW':'mean','Load_MW':'mean'}).reset_index()

In [111]:
df1_2[df1_2['Dispatch_MW'] ==0].to_csv(r'D:\Analysis\Battery Pattern\Raw Data\VIC_t3.csv')

In [97]:
df1_2[df1_2['Load_MW'] ==0].to_csv(r'D:\Analysis\Battery Pattern\Raw Data\t4.csv')